In this playbook I built a LLM compiler that parses pages and provide back the news.

First we crawl the url page to collect news related links

In [24]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_contents, fetch_website_links
from IPython.display import Markdown, display, update_display


In [10]:
load_dotenv(override=True)

api_key = os.getenv('LLM_API_KEY')
model = os.getenv('LLM_MODEL', 'gpt-4.1-mini')
base_url = os.getenv('LLM_BASE_URL')

if not api_key:
    raise ValueError('Set LLM_API_KEY in .env first')

if not base_url:
    raise ValueError('Set LLM_BASE_URL in .env first')
model

'gpt-4.1-mini'

In [11]:
ollama = OpenAI(base_url=base_url, api_key=api_key)
# links = fetch_website_links("https://kubernetes.io")
# links


In [12]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a newsletter related to the main product
of the webpage.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "news page", "url": "https://full.url/goes/here/news"},
        {"type": "blog page", "url": "https://another.full.url/blog"}
    ]
}
"""

In [13]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for latest news, blogs and issues. 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [14]:
print(get_links_user_prompt("https://kubernetes.io"))


Here is the list of links on the website https://kubernetes.io -
Please decide which of these are relevant web links for latest news, blogs and issues. 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/docs/home/
/blog/
/training/
/careers/
/partners/
/community/
#
/releases
https://kubernetes.io
https://v1-35.docs.kubernetes.io
https://v1-34.docs.kubernetes.io
https://v1-33.docs.kubernetes.io
https://v1-32.docs.kubernetes.io
#
/bn/
/zh-cn/
/fr/
/de/
/hi/
/id/
/it/
/ja/
/ko/
/fa/
/pl/
/pt-br/
/ru/
/es/
/uk/
/vi/
https://events.linuxfoundation.org/kubecon-cloudnativecon-india/
https://events.linuxfoundation.org/kubecon-cloudnativecon-india/register/?utm_source=kubernetes&utm_medium=homepage&utm_campaign=KubeCon-India-2026&utm_content=hero
/docs/tutorials/kubernetes-basics/
/docs/concepts/overview/
https://queue.acm.org/detail.cfm?id=2898444
/releases/download/
https://events.linuxfoundation

In [15]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {model}")
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links
    

In [16]:
select_relevant_links("https://kubernetes.io")

Selecting relevant links for https://kubernetes.io by calling gpt-4.1-mini
Found 6 relevant links


{'links': [{'type': 'blog page', 'url': 'https://kubernetes.io/blog/'},
  {'type': 'news/releases page', 'url': 'https://kubernetes.io/releases'},
  {'type': 'community discussion forum',
   'url': 'https://discuss.kubernetes.io'},
  {'type': 'events KubeCon India',
   'url': 'https://events.linuxfoundation.org/kubecon-cloudnativecon-india/'},
  {'type': 'events KubeCon Europe',
   'url': 'https://events.linuxfoundation.org/kubecon-cloudnativecon-europe-2026/'},
  {'type': 'events KubeCon North America',
   'url': 'https://events.linuxfoundation.org/kubecon-cloudnativecon-north-america-2026/'}]}

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
news_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a technology website
and creates summary of the latest news, blogs and issue pertaining to the technology.
Respond in markdown without code blocks.
"""

In [19]:
def get_news_prompt(url):
    user_prompt = f"""
You are looking at a page of a technology website.
Here are the contents of its landing page and other relevant pages;
use this information to provide list of the latest news and blogs in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_news_prompt("https://kubernetes.io")

Selecting relevant links for https://kubernetes.io by calling gpt-4.1-mini
Found 7 relevant links


"\nYou are looking at a page of a technology website.\nHere are the contents of its landing page and other relevant pages;\nuse this information to provide list of the latest news and blogs in markdown without code blocks.\n\n\n## Landing Page:\n\nKubernetes\n\nKubernetes\nDocumentation\nKubernetes Blog\nTraining\nCareers\nPartners\nCommunity\nVersions\nRelease Information\nv1.36\nv1.35\nv1.34\nv1.33\nv1.32\nEnglish\nবাংলা (Bengali)\n中文 (Chinese)\nFrançais (French)\nDeutsch (German)\nहिन्दी (Hindi)\nBahasa Indonesia (Indonesian)\nItaliano (Italian)\n日本語 (Japanese)\n한국어 (Korean)\nفارسی (Persian)\nPolski (Polish)\nPortuguês (Portuguese)\nРусский (Russian)\nEspañol (Spanish)\nУкраїнська (Ukrainian)\nTiếng Việt (Vietnamese)\nKubeCon + CloudNativeCon India 2026\nJoin us for two days of incredible opportunities to collaborate, learn and share with the cloud native community.\nBuy your ticket now! 18 - 19 June | Mumbai, India\nProduction-Grade Container Orchestration\nLearn Kubernetes Basics\

In [21]:
def create_news(url):
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": news_system_prompt},
            {"role": "user", "content": get_news_prompt(url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
def stream_news(url):
    stream = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": news_system_prompt},
            {"role": "user", "content": get_news_prompt(url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_news("https://www.docker.com")

Selecting relevant links for https://www.docker.com by calling gpt-4.1-mini
Found 9 relevant links


# Latest News and Blogs from Docker

### News and Product Highlights
- **Docker Sandboxes**: New isolated environments designed specifically for coding AI agents.
- **AI Governance**: A new feature to govern agents and Claws across teams, enhancing control and compliance.
- **Docker Model Runner**: Facilitates local-first Large Language Model (LLM) inference, making AI deployment more accessible.
- **Docker MCP Catalog and Toolkit**: Tools to connect and manage MCP (multi-cloud processors) tools efficiently.
- **Application Security Enhancements**:
  - Docker Hardened Images for shipping secure, enterprise-ready container images.
  - Docker Scout to simplify and secure the software supply chain.
- **Application Development Tools**:
  - Docker Desktop for containerizing applications.
  - Docker Hub for discovering and sharing container images.
  - Docker Offload to overcome local resource constraints.
  - Docker Build Cloud for speeding up image builds.
  - Testcontainers Desktop for local testing using real dependencies.

### Latest Blog Post
- **Docker MCP for AI Agents: Real-World Developer Setup**  
  This blog post covers practical setups and usage of Docker MCP in managing AI agents, focusing on real-world developer scenarios to streamline AI workflows using Docker’s new AI and agent tools.

### Additional Resources
- Docker offers extensive documentation, training, and community engagement channels for developers to deepen their Docker proficiency.
- Customers share inspirational stories highlighting Docker's impact and utility in enterprise and individual projects.
- Docker provides flexible pricing plans:
  - **Docker Personal** (Free) for individual developers with essential tools.
  - **Docker Pro** ($9-11/month) for professionals offering enhanced features and support.

This summary highlights Docker’s recent innovations focused on AI and agent development, improvements in application security, development efficiency, and practical guidance through their latest blog and documentation.